In [6]:
import pandas as pd
import sqlite3

df = pd.read_csv("../../data/Metro_Interstate_Traffic_Volume.csv")

print("Dataset shape:", df.shape)
print(df.columns.tolist())

df.head()

Dataset shape: (48204, 9)
['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'weather_description', 'date_time', 'traffic_volume']


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [7]:
# Testing area 
# import os

# print("Current folder:", os.getcwd())
# print("Files here:", os.listdir())
# print("Parent folder contents:", os.listdir(".."))

In [12]:
# Task 
# Task 1: SQL-Based Traffic Analysis 
# Using the Metro Interstate Traffic Volume dataset: 
import sqlite3
import os

# Create the SQL folder if it does not already exist
os.makedirs("../sql", exist_ok=True)

# Create/connect to SQLite database
conn = sqlite3.connect("../sql/traffic.db")

# Load dataframe into SQLite
df.to_sql(
    "traffic",
    conn,
    if_exists="replace",
    index=False
)

print("Traffic data loaded into SQLite successfully.")

Traffic data loaded into SQLite successfully.


In [14]:
#Verify if the data has been loaded correctly 
query = """
SELECT COUNT(*) AS total_rows
FROM traffic;
"""

pd.read_sql_query(query, conn)

#Verify if records exist 
pd.read_sql_query(
    "SELECT * FROM traffic LIMIT 5;",
    conn
)


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,None,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,None,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [15]:
#1.2 Analyse annual traffic trends
query_yearly = """
SELECT
    SUBSTR(date_time, 1, 4) AS year,
    SUM(traffic_volume) AS total_traffic_volume
FROM traffic
WHERE SUBSTR(date_time, 1, 4) BETWEEN '2012' AND '2017'
GROUP BY year
ORDER BY year;
"""

yearly_traffic = pd.read_sql_query(query_yearly, conn)
yearly_traffic

,year,total_traffic_volume
0,2012,8208767
1,2013,28177412
2,2014,15731289
3,2015,14181206
4,2016,29494821
5,2017,35428156


In [16]:
yearly_traffic["change"] = yearly_traffic["total_traffic_volume"].diff()

yearly_traffic["percent_change"] = (
    yearly_traffic["total_traffic_volume"]
    .pct_change() * 100
)

yearly_traffic

,year,total_traffic_volume,change,percent_change
0,2012,8208767,NaN,NaN
1,2013,28177412,19968645.0,243.259980
2,2014,15731289,-12446123.0,-44.170568
3,2015,14181206,-1550083.0,-9.853503
4,2016,29494821,15313615.0,107.985280
5,2017,35428156,5933335.0,20.116532


### Explain at least two meaningful observatons about traffic volume trends: 
#### Observations about Traffic Volume Trends across years. 

1. Traffic volume decreased in both 2014 and 2015 after the higher total recorded in 2013, before increasing sharply again in 2016.
2. The highest total traffic volume was recorded in 2017, with 35,428,156 vehicles, representing a further 20.12% increase from 2016.
3. Year-to-year comparisons should be interpreted cautiously because the dataset does not contain complete calendar-year coverage for every year.

In [17]:
#1.3 Analyse temperature around holidays
holiday_query = """
SELECT
    SUBSTR(date_time, 1, 4) AS year,
    holiday,
    AVG(temp) AS avg_temperature,
    AVG(traffic_volume) AS avg_traffic_volume,
    COUNT(*) AS records
FROM traffic
WHERE SUBSTR(date_time, 1, 4) IN ('2015', '2016', '2017')
  AND holiday IN ('New Years Day', 'Labor Day')
GROUP BY year, holiday
ORDER BY year, holiday;
"""

holiday_analysis = pd.read_sql_query(holiday_query, conn)
holiday_analysis

,year,holiday,avg_temperature,avg_traffic_volume,records
0,2015,Labor Day,295.02,973.0,1
1,2016,Labor Day,293.17,1064.0,2
2,2016,New Years Day,265.94,1513.0,2
3,2017,Labor Day,295.54,1026.0,1
4,2017,New Years Day,270.62,798.0,1


### Identify notable year-on-year changes and discuss whether these changes appear relevant to traffic conditions 
#### Holiday Temperature Observations

1. Labor Day temperature decreased from 295.02 K in 2015 to 293.17 K in 2016,
   before increasing to 295.54 K in 2017. Traffic volume remained relatively
   similar across these Labor Day observations.

2. New Year's Day temperature increased from 265.94 K in 2016 to 270.62 K in
   2017, while the observed traffic volume decreased from 1,513 to 798.

3. No New Year's Day record is available for 2015 because the dataset does not
   cover the beginning of 2015. Therefore, a complete three-year comparison
   cannot be made for this holiday.

Overall, these limited holiday observations do not provide enough evidence to
conclude that temperature alone explains changes in traffic volume.

In [ ]:
#Task 2: Descriptive Statistics and Correlation 

In [18]:
# 2.1 Traffic volume statistics
traffic_mean = df["traffic_volume"].mean()
traffic_median = df["traffic_volume"].median()
traffic_std = df["traffic_volume"].std()
traffic_variance = df["traffic_volume"].var()
traffic_range = df["traffic_volume"].max() - df["traffic_volume"].min()

print("Mean:", traffic_mean)
print("Median:", traffic_median)
print("Standard deviation:", traffic_std)
print("Variance:", traffic_variance)
print("Range:", traffic_range)

Mean: 3259.8183553232097
Median: 3380.0
Standard deviation: 1986.8606704463589
Variance: 3947615.3237665547
Range: 7280


### Interpret what these statistics tell you about traffic volume and its variability:
#### Traffic volume statistics Interpretation

The mean traffic volume is approximately **3,259.82**, while the median is **3,380**. Since the mean and median are relatively close, the distribution of traffic volume is not extremely skewed.
The standard deviation is approximately **1,986.86**, indicating that traffic volume varies considerably around the mean.
The variance is approximately **3,947,615.32**, which also reflects a high level of variability in traffic volume.
The range is **7,280**, showing a large difference between the minimum and maximum observed traffic volumes.

Overall, the dataset shows substantial variation in traffic volume across different time periods.

In [19]:
correlation = df["temp"].corr(df["traffic_volume"])

print("Correlation between temperature and traffic volume:", correlation)

Correlation between temperature and traffic volume: 0.13029879817112675


### Interpret the direction & strength of the relationship. Explain why correlation does not necessarily imply causation. 
#### Correlation Analysis

The correlation coefficient between temperature and traffic volume is approximately **0.13**.

1. **Direction:** The relationship is positive, meaning that traffic volume tends to increase slightly as temperature increases.
2. **Strength:** The correlation is very weak, because the coefficient is close to 0.

3. **Causation:** Correlation does not imply causation. Although temperature and traffic volume show a slight positive relationship, this does not mean that higher temperature directly causes higher traffic volume. Other factors such as time of day, day of the week, holidays, and weather conditions may also influence traffic volume.

In [20]:
# Task 3:Probability and Congestion Analysis
# 3.1 Basic probability cal

congestion = df["traffic_volume"] > 5500
clear_weather = df["weather_main"].str.lower() == "clear"

total = len(df)

# 1. Probability of congestion - counts rows where traffic volume is > 5,500
p_congestion = congestion.sum() / total

# 2. Probability of clear weather - count rows where weather is clear 
p_clear = clear_weather.sum() / total

# 3. Probability of congestion AND clear weather - both conditions are true 
p_congestion_and_clear = (congestion & clear_weather).sum() / total

print("P(Congestion):", p_congestion)
print("P(Clear weather):", p_clear)
print("P(Congestion AND Clear):", p_congestion_and_clear)

P(Congestion): 0.1472906812712638
P(Clear weather): 0.277798522944154
P(Congestion AND Clear): 0.03657372832130114


In [21]:
# 3.2 Conditional probability
# Define high temperature
high_temp = df["temp"] > 292

# P(Clear | Congestion)
p_clear_given_congestion = (
    (congestion & clear_weather).sum() / congestion.sum()
)

# P(High Temperature | Congestion)
p_high_temp_given_congestion = (
    (congestion & high_temp).sum() / congestion.sum()
)

print("P(Clear | Congestion):", p_clear_given_congestion)
print("P(High Temperature | Congestion):", p_high_temp_given_congestion)

P(Clear | Congestion): 0.2483098591549296
P(High Temperature | Congestion): 0.26295774647887327


In [22]:
# Determine weather conditions and congestion to be considered independent 

p_if_independent = p_congestion * p_clear

print("P(Congestion AND Clear):", p_congestion_and_clear)
print("P(Congestion) x P(Clear):", p_if_independent)

P(Congestion AND Clear): 0.03657372832130114
P(Congestion) x P(Clear): 0.04091713370059525


In [23]:
# cal the Odds ratio of congestion in clear vs. cloudy weather 
# Define cloudy weather
cloudy_weather = df["weather_main"].str.lower() == "clouds"

# Counts for Clear weather
clear_congestion = (congestion & clear_weather).sum()
clear_no_congestion = ((~congestion) & clear_weather).sum()

# Counts for Cloudy weather
cloudy_congestion = (congestion & cloudy_weather).sum()
cloudy_no_congestion = ((~congestion) & cloudy_weather).sum()

# Odds of congestion
odds_clear = clear_congestion / clear_no_congestion
odds_cloudy = cloudy_congestion / cloudy_no_congestion

# Odds ratio
odds_ratio = odds_clear / odds_cloudy

print("Odds of congestion in Clear weather:", odds_clear)
print("Odds of congestion in Cloudy weather:", odds_cloudy)
print("Odds Ratio (Clear vs Cloudy):", odds_ratio)


Odds of congestion in Clear weather: 0.15161678706570347
Odds of congestion in Cloudy weather: 0.20617244670696785
Odds Ratio (Clear vs Cloudy): 0.7353882125733118


### Conclusion with a short explanation of what the probability analysis suggests about RS between weather & congestion. 
#### Probability Analysis

- **P(Congestion) = 0.1473**  
  Approximately **14.73%** of the observations are classified as congested, where traffic volume is greater than 5,500.
- **P(Clear Weather) = 0.2778**  
  Approximately **27.78%** of the observations occurred during clear weather.
- **P(Congestion AND Clear Weather) = 0.0366**  
  Approximately **3.66%** of all observations experienced both congestion and clear weather.
- **P(Clear Weather | Congestion) = 0.2483**  
  Among congested observations, approximately **24.83%** occurred during clear weather.
- **P(High Temperature | Congestion) = 0.2630**  
  Among congested observations, approximately **26.30%** occurred when the temperature was above 292 K.

#### Independence Check

If congestion and clear weather were independent:
P(Congestion) × P(Clear) = **0.0409**

However:
P(Congestion AND Clear) = **0.0366**
Since these values are different, the results suggest that congestion and clear weather are **not independent** in this dataset.

#### Odds Ratio

- Odds of congestion during clear weather = **0.1516**
- Odds of congestion during cloudy weather = **0.2062**
- Odds Ratio (Clear vs Cloudy) = **0.7354**

Since the odds ratio is less than 1, congestion has lower odds during clear weather than during cloudy weather in this dataset.
Overall, weather appears to have some association with congestion, although these probability results alone do not establish a causal relationship.

In [ ]:
# PowerBI Traffic Intelligence Dashboard  